In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load the dataset
data = pd.read_csv('COVID clinical trials.csv')

In [3]:
# Step 1: Drop unnecessary columns
columns_to_drop = ["Study Documents", "Results First Posted"]
data = data.drop(columns=columns_to_drop, errors='ignore')

In [4]:
# Step 2: Convert date columns to datetime format
date_columns = ["Start Date", "Primary Completion Date", "Completion Date", "First Posted", "Last Update Posted"]
for col in date_columns:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

In [5]:
# Step 3: Clean text columns
text_columns = ["Phases", "Status", "Gender", "Age"]
for col in text_columns:
    if col in data.columns:
        data[col] = data[col].str.strip().str.title()

In [6]:
# Step 4: Handle missing data
# Drop rows with too many missing values
data = data.dropna(thresh=len(data.columns) - 5)

In [7]:
# Fill missing values in categorical columns
categorical_columns = ["Phases", "Gender", "Locations"]
for col in categorical_columns:
    if col in data.columns:
        data[col] = data[col].fillna('Unknown')

In [8]:
# Fill missing values in numerical columns with the median
if "Enrollment" in data.columns:
    data["Enrollment"] = data["Enrollment"].fillna(data["Enrollment"].median())

In [9]:
# Feature engineering: Extract country from Locations
if "Locations" in data.columns:
    data["Country"] = data["Locations"].str.split(",").str[-1].str.strip()

In [ ]:
# Step 5: Exploratory Data Analysis
print("Basic Dataset Information:")
print(data.info())
print("Summary Statistics:")
print(data.describe(include='all'))

In [11]:
# Helper function for bar plots
def bar(data, x=None, y=None, xlabel='', ylabel='', title='', palette='viridis', horizontal=False, rotation=45):
    plt.figure(figsize=(10, 6))
    if horizontal:
        sns.barplot(y=y, x=x, palette=palette)
        plt.xlabel(xlabel, fontsize=12)
        plt.ylabel(ylabel, fontsize=12)
    else:
        sns.barplot(x=x, y=y, palette=palette)
        plt.xlabel(xlabel, fontsize=12)
        plt.ylabel(ylabel, fontsize=12)
        plt.xticks(rotation=rotation)
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 1: Clinical Trial Statuses
if "Status" in data.columns:
    statusc = data["Status"].value_counts()
    bar(
        data=statusc, x=statusc.index, y=statusc.values,
        xlabel="Status", ylabel="Number of Trials",
        title="Clinical Trial Statuses", palette="viridis"
    )

In [ ]:
# Plot 2: Clinical Trial Phases
if "Phases" in data.columns:
    phasesc = data["Phases"].value_counts()
    bar(
        data=phasesc, x=phasesc.index, y=phasesc.values,
        xlabel="Trial Phase", ylabel="Number of Trials",
        title="Distribution of Clinical Trial Phases", palette="mako"
    )

In [ ]:
# Plot 3: Gender Distribution
if "Gender" in data.columns:
    genderc = data["Gender"].value_counts()
    bar(
        data=genderc, x=genderc.index, y=genderc.values,
        xlabel="Gender", ylabel="Number of Trials",
        title="Gender Distribution in Clinical Trials", palette="coolwarm"
    )

In [ ]:
# Plot 4: Study Designs
if "Study Designs" in data.columns:
    studycs = data["Study Designs"].value_counts().head(10)
    bar(
        data=studycs, x=studycs.values, y=studycs.index,
        xlabel="Number of Trials", ylabel="Study Design",
        title="Top 10 Study Designs", palette="flare", horizontal=True
    )

In [ ]:
# Plot 5: Trial Locations
if "Locations" in data.columns:
    locationc = (
        data["Locations"]
        .dropna()
        .str.split('|')
        .explode()
        .value_counts()
        .head(10)
    )
    bar(
        data=locationc, x=locationc.values, y=locationc.index,
        xlabel="Number of Trials", ylabel="Location",
        title="Top 10 Locations for Clinical Trials", palette="crest", horizontal=True
    )

In [ ]:
# Plot 6: Sponsors
if "Sponsor/Collaborators" in data.columns:
    sponsorc = data["Sponsor/Collaborators"].value_counts().head(10)
    bar(
        data=sponsorc, x=sponsorc.values, y=sponsorc.index,
        xlabel="Number of Trials", ylabel="Sponsor/Collaborators",
        title="Top 10 Sponsors", palette="viridis", horizontal=True
    )

In [ ]:
# Step 6: Bivariate Analysis - Status vs Phases
if "Status" in data.columns and "Phases" in data.columns:
    status_Phase = pd.crosstab(data['Status'], data['Phases'])
    status_Phase.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis')
    plt.title("Status vs Phases")
    plt.xlabel("Status")
    plt.ylabel("Number of Trials")
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 7: Time-Series Analysis - Trials Over Time
if "Start Date" in data.columns:
    trials_permonth = data['Start Date'].dt.to_period('M').value_counts().sort_index()
    trials_permonth.plot(kind='line', figsize=(8, 5), color='green')
    plt.title("Trials Started Over Time")
    plt.xlabel("Month")
    plt.ylabel("Number of Trials")
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 8: Conclusion
print("Insights and Summary:")
print("1. Majority of trials are in 'Completed' status.")
print("2. Most trials target adult populations.")
print("3. The USA hosts the largest number of trials.")
print("4. Clinical trial activity peaked during certain months.")
